In [1]:
%pip install -q numpy pandas matplotlib pillow opencv-python pyqt5 tifffile openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
%matplotlib qt
%matplotlib tk

In [5]:
#3 Imports, paths, tray mapping and settings

from pathlib import Path
from collections import defaultdict
import json
import re
import shutil
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

from PIL import Image, ImageOps
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\rahma\Downloads\RINA_Internship_Analysis"
)

THIRD_TRIAL_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "Third Trial"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "Third Trial"
)

# ============================================================
# IMAGE AND EXPERIMENT SETTINGS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
}

REQUIRED_BANDS = [
    "D",
    "MS_G",
    "MS_R",
    "MS_RE",
    "MS_NIR",
]

EXPECTED_TRAYS = list(range(1, 13))

# True means a day is analysed only when all 12 trays have
# complete RGB, Green, Red, Red Edge and NIR images.
REQUIRE_COMPLETE_DAY = True

# Heat trays:
# Days 1–2 = inside
# Days 3–4 = outside
# Days 5–6 = inside
# Days 7–8 = outside
HEAT_STARTS_INSIDE = True

# ============================================================
# TRAY GRID
# ============================================================

ROWS = 7
COLS = 10
EXPECTED_CELLS = ROWS * COLS

CELL_SIZE = 120

STANDARD_WIDTH = COLS * CELL_SIZE
STANDARD_HEIGHT = ROWS * CELL_SIZE

INNER_MARGIN_RATIO = 0.12

# ============================================================
# CROPPING SETTINGS
# ============================================================

# Keep False after successful cropping.
# Change to True only when you need to redo every crop.
REDO_CROPS = False

# Reuse saved crop coordinates when available.
USE_SAVED_CROP_POINTS = True

# ============================================================
# RGB GREEN-DETECTION SETTINGS
# ============================================================

GREEN_H_MIN = 25
GREEN_H_MAX = 100
GREEN_S_MIN = 35
GREEN_V_MIN = 25

EXCESS_GREEN_MIN = 15

# A tray cell is considered visibly emerged when at least
# this percentage of its inner area is detected as green.
EMERGED_GREEN_COVER_PCT = 0.25

# ============================================================
# MULTISPECTRAL SETTINGS
# ============================================================

NDVI_VEGETATION_THRESHOLD = 0.15

# ============================================================
# CHECK INPUT LOCATION
# ============================================================

if not THIRD_TRIAL_DIR.exists():
    raise FileNotFoundError(
        "Third Trial folder was not found:\n"
        f"{THIRD_TRIAL_DIR}\n\n"
        "Expected example:\n"
        r"C:\Users\rahma\Downloads\RINA_Internship_Analysis"
        r"\dataset\Third Trial\Day 1\Tray 1"
    )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# THIRD-TRIAL TRAY DESIGN
# ============================================================

tray_design_df = pd.DataFrame(
    [
        {
            "tray_no": 1,
            "microbe_status": "No Microbe",
            "stress_type": "Ideal",
            "assigned_environment": "Inside",
        },
        {
            "tray_no": 2,
            "microbe_status": "No Microbe",
            "stress_type": "Ideal",
            "assigned_environment": "Outside",
        },
        {
            "tray_no": 3,
            "microbe_status": "No Microbe",
            "stress_type": "Moisture",
            "assigned_environment": "Outside",
        },
        {
            "tray_no": 4,
            "microbe_status": "No Microbe",
            "stress_type": "Heat",
            "assigned_environment": "Alternating",
        },
        {
            "tray_no": 5,
            "microbe_status": "Microbe",
            "stress_type": "Moisture",
            "assigned_environment": "Inside",
        },
        {
            "tray_no": 6,
            "microbe_status": "Microbe",
            "stress_type": "Heat",
            "assigned_environment": "Alternating",
        },
        {
            "tray_no": 7,
            "microbe_status": "Microbe",
            "stress_type": "Ideal",
            "assigned_environment": "Outside",
        },
        {
            "tray_no": 8,
            "microbe_status": "Microbe",
            "stress_type": "Heat",
            "assigned_environment": "Alternating",
        },
        {
            "tray_no": 9,
            "microbe_status": "Microbe",
            "stress_type": "Ideal",
            "assigned_environment": "Inside",
        },
        {
            "tray_no": 10,
            "microbe_status": "Microbe",
            "stress_type": "Moisture",
            "assigned_environment": "Outside",
        },
        {
            "tray_no": 11,
            "microbe_status": "No Microbe",
            "stress_type": "Heat",
            "assigned_environment": "Alternating",
        },
        {
            "tray_no": 12,
            "microbe_status": "No Microbe",
            "stress_type": "Moisture",
            "assigned_environment": "Inside",
        },
    ]
)

tray_design_df["tray"] = (
    "Tray "
    + tray_design_df["tray_no"].astype(str)
)

tray_design_df["treatment_group"] = (
    tray_design_df["microbe_status"]
    + " | "
    + tray_design_df["stress_type"]
)

tray_design_df["design_group"] = (
    tray_design_df["microbe_status"]
    + " | "
    + tray_design_df["stress_type"]
    + " | "
    + tray_design_df["assigned_environment"]
)


def actual_environment(
    day_number,
    stress_type,
    assigned_environment,
):
    """
    Return the real location of a tray on a particular day.

    Ideal and moisture trays remain in their assigned location.
    Heat trays alternate every two days.
    """

    if stress_type != "Heat":
        return assigned_environment

    two_day_block = (
        int(day_number) - 1
    ) // 2

    inside_block = (
        two_day_block % 2 == 0
    )

    if not HEAT_STARTS_INSIDE:
        inside_block = not inside_block

    if inside_block:
        return "Inside"

    return "Outside"


def watering_regime(stress_type):
    """
    Describe the watering treatment.
    """

    if stress_type in {
        "Ideal",
        "Heat",
    }:
        return "Always watered"

    return "Moisture treatment"


print("Third Trial input folder:")
print(THIRD_TRIAL_DIR)

print("\nThird Trial output folder:")
print(OUTPUT_ROOT)

print("\nThird Trial tray design:")
display(tray_design_df)

Third Trial input folder:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\dataset\Third Trial

Third Trial output folder:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\Third Trial

Third Trial tray design:


,tray_no,microbe_status,stress_type,assigned_environment,tray,treatment_group,design_group
0,1,No Microbe,Ideal,Inside,Tray 1,No Microbe | Ideal,No Microbe | Ideal | Inside
1,2,No Microbe,Ideal,Outside,Tray 2,No Microbe | Ideal,No Microbe | Ideal | Outside
2,3,No Microbe,Moisture,Outside,Tray 3,No Microbe | Moisture,No Microbe | Moisture | Outside
3,4,No Microbe,Heat,Alternating,Tray 4,No Microbe | Heat,No Microbe | Heat | Alternating
4,5,Microbe,Moisture,Inside,Tray 5,Microbe | Moisture,Microbe | Moisture | Inside
5,6,Microbe,Heat,Alternating,Tray 6,Microbe | Heat,Microbe | Heat | Alternating
6,7,Microbe,Ideal,Outside,Tray 7,Microbe | Ideal,Microbe | Ideal | Outside
7,8,Microbe,Heat,Alternating,Tray 8,Microbe | Heat,Microbe | Heat | Alternating
8,9,Microbe,Ideal,Inside,Tray 9,Microbe | Ideal,Microbe | Ideal | Inside
9,10,Microbe,Moisture,Outside,Tray 10,Microbe | Moisture,Microbe | Moisture | Outside


In [9]:
#4 Discover all image files and build the manifest

def normalise_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(value).lower(),
    ).strip()


def natural_key(value):
    return [
        int(part)
        if part.isdigit()
        else part.casefold()
        for part in re.split(
            r"(\d+)",
            str(value),
        )
    ]


def extract_day_number(value):
    match = re.search(
        r"\bday\s*(\d+)\b",
        normalise_name(value),
    )

    if match:
        return int(match.group(1))

    return None


def extract_tray_number(value):
    match = re.search(
        r"\btray\s*(\d+)\b",
        normalise_name(value),
    )

    if match:
        return int(match.group(1))

    return None


def locate_day_and_tray(image_path):
    """
    Extract the day and tray number from the image's folders.
    """

    day_number = None
    tray_number = None

    relative_parts = image_path.relative_to(
        THIRD_TRIAL_DIR
    ).parts[:-1]

    for part in relative_parts:
        possible_day = extract_day_number(part)
        possible_tray = extract_tray_number(part)

        if possible_day is not None:
            day_number = possible_day

        if possible_tray in EXPECTED_TRAYS:
            tray_number = possible_tray

    if (
        day_number is None
        or tray_number is None
    ):
        return None

    return day_number, tray_number


def parse_band_and_sequence(image_path):
    """
    Detect RGB and multispectral band information from a filename.

    Expected examples:
    DJI_20260627101229_0001_D.JPG
    DJI_20260627101230_0001_MS_G.TIF
    DJI_20260627101230_0001_MS_R.TIF
    DJI_20260627101230_0001_MS_RE.TIF
    DJI_20260627101230_0001_MS_NIR.TIF
    """

    stem = image_path.stem.upper()

    standard_match = re.match(
        r"^.+?_(?P<sequence>\d{3,})_"
        r"(?P<band>MS_NIR|MS_RE|MS_R|MS_G|D|F)$",
        stem,
        flags=re.IGNORECASE,
    )

    if standard_match:
        band = (
            standard_match
            .group("band")
            .upper()
        )

        if band == "F":
            return None

        return {
            "sequence": standard_match.group(
                "sequence"
            ),
            "band": band,
        }

    suffix_patterns = [
        (
            "MS_NIR",
            r"(?:^|[_\-\s])MS[_\-\s]?NIR$",
        ),
        (
            "MS_RE",
            r"(?:^|[_\-\s])MS[_\-\s]?RE$",
        ),
        (
            "MS_R",
            r"(?:^|[_\-\s])MS[_\-\s]?R$",
        ),
        (
            "MS_G",
            r"(?:^|[_\-\s])MS[_\-\s]?G$",
        ),
        (
            "D",
            r"(?:^|[_\-\s])D$",
        ),
        (
            "F",
            r"(?:^|[_\-\s])F$",
        ),
    ]

    for band, pattern in suffix_patterns:
        if re.search(
            pattern,
            stem,
            flags=re.IGNORECASE,
        ):
            if band == "F":
                return None

            without_band = re.sub(
                pattern,
                "",
                stem,
                flags=re.IGNORECASE,
            ).rstrip("_- ")

            numbers = re.findall(
                r"\d+",
                without_band,
            )

            if numbers:
                sequence = numbers[-1]
            else:
                sequence = without_band

            return {
                "sequence": sequence,
                "band": band,
            }

    return None


# ============================================================
# FIND ALL IMAGES
# ============================================================

all_images = sorted(
    [
        path
        for path in THIRD_TRIAL_DIR.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ],
    key=lambda path: natural_key(
        str(path)
    ),
)

print(
    "Total image files found:",
    len(all_images),
)

if not all_images:
    raise RuntimeError(
        "No image files were found inside:\n"
        f"{THIRD_TRIAL_DIR}"
    )

# ============================================================
# GROUP RGB AND MULTISPECTRAL FILES
# ============================================================

grouped = defaultdict(dict)
unmatched_records = []

for image_path in all_images:
    location = locate_day_and_tray(
        image_path
    )

    metadata = parse_band_and_sequence(
        image_path
    )

    if (
        location is None
        or metadata is None
    ):
        unmatched_records.append(
            {
                "relative_path": str(
                    image_path.relative_to(
                        THIRD_TRIAL_DIR
                    )
                ),
                "day_tray_found": (
                    location is not None
                ),
                "filename_matched": (
                    metadata is not None
                ),
            }
        )
        continue

    day_number, tray_number = location

    sequence = str(
        metadata["sequence"]
    )

    band = metadata["band"]

    group_key = (
        day_number,
        tray_number,
        sequence,
    )

    grouped[group_key].setdefault(
        band,
        image_path,
    )

# ============================================================
# CREATE RAW MANIFEST
# ============================================================

manifest_records = []

for (
    day_number,
    tray_number,
    sequence,
), band_paths in sorted(
    grouped.items(),
    key=lambda item: (
        item[0][0],
        item[0][1],
        natural_key(item[0][2]),
    ),
):
    missing_bands = [
        band
        for band in REQUIRED_BANDS
        if band not in band_paths
    ]

    preferred_path = (
        band_paths.get("D")
        or band_paths.get("MS_NIR")
        or next(
            iter(
                band_paths.values()
            )
        )
    )

    capture_id = re.sub(
        r"_(D|MS_G|MS_R|MS_RE|MS_NIR)$",
        "",
        preferred_path.stem,
        flags=re.IGNORECASE,
    )

    manifest_records.append(
        {
            "day_order": day_number,
            "day": f"Day {day_number}",
            "tray_no": tray_number,
            "tray": f"Tray {tray_number}",
            "sequence": sequence,
            "capture_id": capture_id,
            "source_d": str(
                band_paths.get(
                    "D",
                    "",
                )
            ),
            "source_ms_g": str(
                band_paths.get(
                    "MS_G",
                    "",
                )
            ),
            "source_ms_r": str(
                band_paths.get(
                    "MS_R",
                    "",
                )
            ),
            "source_ms_re": str(
                band_paths.get(
                    "MS_RE",
                    "",
                )
            ),
            "source_ms_nir": str(
                band_paths.get(
                    "MS_NIR",
                    "",
                )
            ),
            "bands_found": ", ".join(
                sorted(
                    band_paths.keys()
                )
            ),
            "missing_bands": ", ".join(
                missing_bands
            ),
            "complete_set": (
                len(missing_bands) == 0
            ),
        }
    )

raw_manifest_df = pd.DataFrame(
    manifest_records
)

if raw_manifest_df.empty:
    print("\nFirst unmatched files:")

    display(
        pd.DataFrame(
            unmatched_records
        ).head(30)
    )

    raise RuntimeError(
        "No valid five-band image groups were discovered. "
        "Check the displayed filenames and folder structure."
    )

raw_manifest_df = (
    raw_manifest_df
    .sort_values(
        [
            "day_order",
            "tray_no",
            "sequence",
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# KEEP COMPLETE FIVE-BAND SETS
# ============================================================

complete_sets_df = raw_manifest_df[
    raw_manifest_df["complete_set"]
].copy()

# Keep one complete image group per Day/Tray.
selected_before_day_filter_df = (
    complete_sets_df
    .sort_values(
        [
            "day_order",
            "tray_no",
            "sequence",
        ]
    )
    .drop_duplicates(
        subset=[
            "day_order",
            "tray_no",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ============================================================
# CHECK WHETHER EACH DAY HAS ALL 12 TRAYS
# ============================================================

day_completeness_df = (
    selected_before_day_filter_df
    .groupby(
        [
            "day_order",
            "day",
        ],
        as_index=False,
    )
    .agg(
        complete_trays=(
            "tray_no",
            "nunique",
        )
    )
)

day_completeness_df[
    "expected_trays"
] = len(EXPECTED_TRAYS)

day_completeness_df[
    "complete_day"
] = (
    day_completeness_df[
        "complete_trays"
    ]
    == len(EXPECTED_TRAYS)
)

if REQUIRE_COMPLETE_DAY:
    accepted_days = (
        day_completeness_df.loc[
            day_completeness_df[
                "complete_day"
            ],
            "day_order",
        ]
        .tolist()
    )

    analysis_manifest_df = (
        selected_before_day_filter_df[
            selected_before_day_filter_df[
                "day_order"
            ].isin(accepted_days)
        ]
        .copy()
    )

else:
    accepted_days = sorted(
        selected_before_day_filter_df[
            "day_order"
        ]
        .unique()
        .tolist()
    )

    analysis_manifest_df = (
        selected_before_day_filter_df
        .copy()
    )

# ============================================================
# ADD EXPERIMENTAL DESIGN INFORMATION
# ============================================================

analysis_manifest_df = (
    analysis_manifest_df
    .merge(
        tray_design_df,
        on=[
            "tray_no",
            "tray",
        ],
        how="left",
    )
    .sort_values(
        [
            "day_order",
            "tray_no",
        ]
    )
    .reset_index(drop=True)
)

analysis_manifest_df[
    "actual_environment"
] = analysis_manifest_df.apply(
    lambda row: actual_environment(
        row["day_order"],
        row["stress_type"],
        row["assigned_environment"],
    ),
    axis=1,
)

analysis_manifest_df[
    "watering_regime"
] = analysis_manifest_df[
    "stress_type"
].map(watering_regime)

# ============================================================
# IDENTIFY MISSING DAY/TRAY SETS
# ============================================================

all_discovered_days = sorted(
    raw_manifest_df[
        "day_order"
    ]
    .unique()
    .tolist()
)

expected_pairs = {
    (
        day_number,
        tray_number,
    )
    for day_number in all_discovered_days
    for tray_number in EXPECTED_TRAYS
}

found_complete_pairs = set(
    zip(
        selected_before_day_filter_df[
            "day_order"
        ],
        selected_before_day_filter_df[
            "tray_no"
        ],
    )
)

missing_pairs = sorted(
    expected_pairs
    - found_complete_pairs
)

print(
    "\nRaw image groups:",
    len(raw_manifest_df),
)

print(
    "Complete five-band groups:",
    len(complete_sets_df),
)

print(
    "Selected unique Day/Tray groups:",
    len(selected_before_day_filter_df),
)

print(
    "Accepted days:",
    accepted_days,
)

print(
    "Final groups for analysis:",
    len(analysis_manifest_df),
)

print("\nDay completeness:")
display(day_completeness_df)

if missing_pairs:
    print(
        "\nMissing complete Day/Tray combinations:"
    )

    for (
        day_number,
        tray_number,
    ) in missing_pairs:
        print(
            f"Day {day_number}, "
            f"Tray {tray_number}"
        )

duplicate_complete_df = (
    complete_sets_df
    .groupby(
        [
            "day_order",
            "day",
            "tray_no",
            "tray",
        ],
        as_index=False,
    )
    .size()
)

duplicate_complete_df = (
    duplicate_complete_df[
        duplicate_complete_df[
            "size"
        ] > 1
    ]
)

if not duplicate_complete_df.empty:
    print(
        "\nDay/Tray folders with multiple complete image groups:"
    )

    display(
        duplicate_complete_df
    )

if analysis_manifest_df.empty:
    raise RuntimeError(
        "No days were accepted for analysis. "
        "With REQUIRE_COMPLETE_DAY=True, each analysed day "
        "must contain complete files for all 12 trays. "
        "Fix the missing files or change "
        "REQUIRE_COMPLETE_DAY=False in Cell 3."
    )

display(
    analysis_manifest_df[
        [
            "day",
            "tray",
            "sequence",
            "microbe_status",
            "stress_type",
            "assigned_environment",
            "actual_environment",
            "watering_regime",
        ]
    ]
)

print("\nCELL COMPLETED.")

Total image files found: 455

Raw image groups: 12
Complete five-band groups: 12
Selected unique Day/Tray groups: 12
Accepted days: [1]
Final groups for analysis: 12

Day completeness:


,day_order,day,complete_trays,expected_trays,complete_day
0,1,Day 1,12,12,True


,day,tray,sequence,microbe_status,stress_type,assigned_environment,actual_environment,watering_regime
0,Day 1,Tray 1,0001,No Microbe,Ideal,Inside,Inside,Always watered
1,Day 1,Tray 2,0003,No Microbe,Ideal,Outside,Outside,Always watered
2,Day 1,Tray 3,0004,No Microbe,Moisture,Outside,Outside,Moisture treatment
3,Day 1,Tray 4,0005,No Microbe,Heat,Alternating,Inside,Always watered
4,Day 1,Tray 5,0006,Microbe,Moisture,Inside,Inside,Moisture treatment
5,Day 1,Tray 6,0007,Microbe,Heat,Alternating,Inside,Always watered
6,Day 1,Tray 7,0008,Microbe,Ideal,Outside,Outside,Always watered
7,Day 1,Tray 8,0009,Microbe,Heat,Alternating,Inside,Always watered
8,Day 1,Tray 9,0010,Microbe,Ideal,Inside,Inside,Always watered
9,Day 1,Tray 10,0011,Microbe,Moisture,Outside,Outside,Moisture treatment



CELL COMPLETED.


In [11]:
#5 Create output folders and helper functions

# ============================================================
# OUTPUT SECTIONS
# ============================================================

S01 = (
    OUTPUT_ROOT
    / "01_Crop_Dual_Reference"
)

S02 = (
    OUTPUT_ROOT
    / "02_Crop_Quality_Check"
)

S03 = (
    OUTPUT_ROOT
    / "03_Cell_Grid_Detection"
)

S04 = (
    OUTPUT_ROOT
    / "04_Visible_Emergence"
)

S05 = (
    OUTPUT_ROOT
    / "05_Treatment_Growth_Visuals"
)

S06 = (
    OUTPUT_ROOT
    / "06_MS_Cell_Grid_Detection"
)

S07 = (
    OUTPUT_ROOT
    / "07_MS_Vegetation_Indices"
)

S08 = (
    OUTPUT_ROOT
    / "08_MS_Treatment_Comparison"
)

S09 = (
    OUTPUT_ROOT
    / "09_Third_Trial_Synthesis"
)

S10 = (
    OUTPUT_ROOT
    / "10_Report_Figure_Package"
)

folders_to_create = [
    S01,
    S02,
    S03,
    S04,
    S05,
    S06,
    S07,
    S08,
    S09,
    S10,

    S01 / "_reports",
    S01 / "_config",
    S01 / "_diagnostics",

    S02 / "_reports",
    S02 / "previews",

    S03 / "_reports",
    S03 / "overlays",

    S04 / "_reports",
    S04 / "overlays",

    S05 / "_reports",
    S05 / "charts",

    S06 / "_reports",
    S06 / "overlays",

    S07 / "_reports",
    S07 / "heatmaps",

    S08 / "_reports",
    S08 / "charts",

    S09 / "_reports",
    S09 / "charts",
    S09 / "manual_validation",

    S10 / "figures",
    S10 / "reports",
]

for folder in folders_to_create:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

# ============================================================
# SAVE INITIAL MANIFESTS
# ============================================================

raw_manifest_df.to_csv(
    S01
    / "_reports"
    / "raw_dataset_manifest.csv",
    index=False,
)

selected_before_day_filter_df.to_csv(
    S01
    / "_reports"
    / "selected_before_day_filter.csv",
    index=False,
)

analysis_manifest_df.to_csv(
    S01
    / "_reports"
    / "selected_analysis_manifest.csv",
    index=False,
)

day_completeness_df.to_csv(
    S01
    / "_reports"
    / "day_completeness.csv",
    index=False,
)

tray_design_df.to_csv(
    S01
    / "_reports"
    / "third_trial_tray_design.csv",
    index=False,
)

# ============================================================
# SAVE ANALYSIS SETTINGS
# ============================================================

settings_path = (
    S01
    / "_config"
    / "analysis_settings.json"
)

with open(
    settings_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "rows": ROWS,
            "columns": COLS,
            "cells_per_tray": (
                EXPECTED_CELLS
            ),
            "standard_width": (
                STANDARD_WIDTH
            ),
            "standard_height": (
                STANDARD_HEIGHT
            ),
            "require_complete_day": (
                REQUIRE_COMPLETE_DAY
            ),
            "heat_starts_inside": (
                HEAT_STARTS_INSIDE
            ),
            "green_h_min": (
                GREEN_H_MIN
            ),
            "green_h_max": (
                GREEN_H_MAX
            ),
            "green_s_min": (
                GREEN_S_MIN
            ),
            "green_v_min": (
                GREEN_V_MIN
            ),
            "excess_green_min": (
                EXCESS_GREEN_MIN
            ),
            "emerged_green_cover_pct": (
                EMERGED_GREEN_COVER_PCT
            ),
            "ndvi_vegetation_threshold": (
                NDVI_VEGETATION_THRESHOLD
            ),
        },
        file,
        indent=2,
    )

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_name(value):
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value),
    ).strip("_")


def read_rgb(path):
    """
    Read an RGB image and apply its EXIF orientation.
    """

    with Image.open(path) as image:
        image = ImageOps.exif_transpose(
            image
        ).convert("RGB")

        return np.asarray(image)


def read_band(path):
    """
    Read one multispectral TIFF band.
    """

    image = tifffile.imread(path)
    image = np.squeeze(image)

    if image.ndim != 2:
        raise ValueError(
            "Expected a single-band image:\n"
            f"{path}"
        )

    return image


def display_stretch(image):
    """
    Convert a multispectral image into an 8-bit image
    for display only.
    """

    values = image.astype(
        np.float32
    )

    valid = values[
        np.isfinite(values)
    ]

    if valid.size == 0:
        return np.zeros(
            values.shape,
            dtype=np.uint8,
        )

    low, high = np.percentile(
        valid,
        [1, 99],
    )

    if high <= low:
        high = low + 1.0

    stretched = (
        values - low
    ) * 255.0 / (
        high - low
    )

    return np.clip(
        stretched,
        0,
        255,
    ).astype(np.uint8)


def order_four_points(points):
    """
    Automatically arrange four clicked points as:
    top-left, top-right, bottom-right, bottom-left.
    """

    points = np.asarray(
        points,
        dtype=np.float32,
    )

    if points.shape != (4, 2):
        raise ValueError(
            "Exactly four x/y points are required."
        )

    ordered = np.zeros(
        (4, 2),
        dtype=np.float32,
    )

    sums = points.sum(axis=1)

    differences = np.diff(
        points,
        axis=1,
    ).ravel()

    ordered[0] = points[
        np.argmin(sums)
    ]

    ordered[2] = points[
        np.argmax(sums)
    ]

    ordered[1] = points[
        np.argmin(differences)
    ]

    ordered[3] = points[
        np.argmax(differences)
    ]

    return ordered


def select_four_corners(
    image,
    title,
    grayscale=False,
):
    """
    Display an image and allow four tray corners to be clicked.
    """

    figure, axis = plt.subplots(
        figsize=(14, 9)
    )

    axis.imshow(
        image,
        cmap=(
            "gray"
            if grayscale
            else None
        ),
    )

    axis.set_title(
        title
        + "\nClick the four outer tray corners in any order."
        + "\nRight-click removes the latest point."
    )

    axis.axis("off")

    clicked = plt.ginput(
        4,
        timeout=0,
        show_clicks=True,
        mouse_add=1,
        mouse_pop=3,
    )

    plt.close(figure)

    if len(clicked) != 4:
        return None

    return order_four_points(
        clicked
    )


def perspective_matrix(points):
    source = order_four_points(
        points
    )

    destination = np.asarray(
        [
            [0, 0],
            [
                STANDARD_WIDTH - 1,
                0,
            ],
            [
                STANDARD_WIDTH - 1,
                STANDARD_HEIGHT - 1,
            ],
            [
                0,
                STANDARD_HEIGHT - 1,
            ],
        ],
        dtype=np.float32,
    )

    return cv2.getPerspectiveTransform(
        source,
        destination,
    )


def warp_to_standard(
    image,
    points,
    interpolation,
):
    """
    Apply perspective correction and resize to 1200 × 840.
    """

    matrix = perspective_matrix(
        points
    )

    return cv2.warpPerspective(
        image,
        matrix,
        (
            STANDARD_WIDTH,
            STANDARD_HEIGHT,
        ),
        flags=interpolation,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


def cell_bounds(
    row_index,
    column_index,
    inner_margin=True,
):
    """
    Return the pixel coordinates for one grid cell.
    """

    x0 = int(
        round(
            column_index
            * STANDARD_WIDTH
            / COLS
        )
    )

    x1 = int(
        round(
            (
                column_index + 1
            )
            * STANDARD_WIDTH
            / COLS
        )
    )

    y0 = int(
        round(
            row_index
            * STANDARD_HEIGHT
            / ROWS
        )
    )

    y1 = int(
        round(
            (
                row_index + 1
            )
            * STANDARD_HEIGHT
            / ROWS
        )
    )

    if inner_margin:
        x_margin = int(
            round(
                (
                    x1 - x0
                )
                * INNER_MARGIN_RATIO
            )
        )

        y_margin = int(
            round(
                (
                    y1 - y0
                )
                * INNER_MARGIN_RATIO
            )
        )

        x0 += x_margin
        x1 -= x_margin
        y0 += y_margin
        y1 -= y_margin

    return x0, y0, x1, y1


def add_grid(axis):
    for column_index in range(
        COLS + 1
    ):
        x_position = (
            column_index
            * STANDARD_WIDTH
            / COLS
        )

        axis.axvline(
            x_position,
            linewidth=0.6,
        )

    for row_index in range(
        ROWS + 1
    ):
        y_position = (
            row_index
            * STANDARD_HEIGHT
            / ROWS
        )

        axis.axhline(
            y_position,
            linewidth=0.6,
        )


def crop_file_paths(
    day_number,
    tray_number,
):
    """
    Return all five output crop paths for one Day/Tray.
    """

    day_folder = (
        S01
        / f"Day {day_number}"
        / f"Tray {tray_number}"
    )

    day_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    prefix = (
        f"Day_{day_number:02d}"
        f"_Tray_{tray_number:02d}"
    )

    return {
        "folder": day_folder,
        "D": (
            day_folder
            / f"{prefix}_D.JPG"
        ),
        "MS_G": (
            day_folder
            / f"{prefix}_MS_G.TIF"
        ),
        "MS_R": (
            day_folder
            / f"{prefix}_MS_R.TIF"
        ),
        "MS_RE": (
            day_folder
            / f"{prefix}_MS_RE.TIF"
        ),
        "MS_NIR": (
            day_folder
            / f"{prefix}_MS_NIR.TIF"
        ),
    }


def save_excel_sheets(
    path,
    sheets,
):
    """
    Save multiple DataFrames into one Excel workbook.
    """

    with pd.ExcelWriter(
        path,
        engine="openpyxl",
    ) as writer:
        for (
            sheet_name,
            dataframe,
        ) in sheets.items():
            dataframe.to_excel(
                writer,
                sheet_name=(
                    safe_name(
                        sheet_name
                    )[:31]
                ),
                index=False,
            )


print("Output folders created successfully.")

print(
    "Standard crop size:",
    STANDARD_WIDTH,
    "×",
    STANDARD_HEIGHT,
)

print(
    "Cells per tray:",
    EXPECTED_CELLS,
)

print(
    "Output root:",
    OUTPUT_ROOT,
)

Output folders created successfully.
Standard crop size: 1200 × 840
Cells per tray: 70
Output root: C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\Third Trial


In [13]:
#6 — Crop every tray using RGB and NIR references

crop_points_path = (
    S01
    / "_config"
    / "crop_points.json"
)

if (
    crop_points_path.exists()
    and USE_SAVED_CROP_POINTS
):
    with open(
        crop_points_path,
        "r",
        encoding="utf-8",
    ) as file:
        crop_points = json.load(
            file
        )
else:
    crop_points = {}

crop_manifest_records = []

for (
    manifest_index,
    row,
) in analysis_manifest_df.iterrows():

    day_number = int(
        row["day_order"]
    )

    tray_number = int(
        row["tray_no"]
    )

    day = row["day"]
    tray = row["tray"]

    crop_key = (
        f"Day {day_number}"
        f"|Tray {tray_number}"
    )

    outputs = crop_file_paths(
        day_number,
        tray_number,
    )

    expected_output_paths = [
        outputs[band]
        for band in REQUIRED_BANDS
    ]

    print(
        f"\n[{manifest_index + 1}/"
        f"{len(analysis_manifest_df)}] "
        f"{day} | {tray} | "
        f"{row['capture_id']}"
    )

    record = {
        "day_order": day_number,
        "day": day,
        "tray_no": tray_number,
        "tray": tray,
        "capture_id": (
            row["capture_id"]
        ),
        "sequence": row["sequence"],
        "status": "",
        "notes": "",
        "crop_d": str(
            outputs["D"]
        ),
        "crop_ms_g": str(
            outputs["MS_G"]
        ),
        "crop_ms_r": str(
            outputs["MS_R"]
        ),
        "crop_ms_re": str(
            outputs["MS_RE"]
        ),
        "crop_ms_nir": str(
            outputs["MS_NIR"]
        ),
    }

    # Skip crops that already exist.
    if (
        all(
            path.exists()
            for path
            in expected_output_paths
        )
        and not REDO_CROPS
    ):
        record[
            "status"
        ] = "SKIPPED_EXISTING"

        record[
            "notes"
        ] = (
            "All five standardised crops "
            "already exist."
        )

        crop_manifest_records.append(
            record
        )

        print(
            "Result: SKIPPED_EXISTING"
        )

        continue

    try:
        source_rgb = read_rgb(
            Path(
                row["source_d"]
            )
        )

        source_nir = read_band(
            Path(
                row["source_ms_nir"]
            )
        )

        if (
            crop_key in crop_points
            and USE_SAVED_CROP_POINTS
        ):
            rgb_points = np.asarray(
                crop_points[
                    crop_key
                ][
                    "rgb_points"
                ],
                dtype=np.float32,
            )

            ms_points = np.asarray(
                crop_points[
                    crop_key
                ][
                    "ms_points"
                ],
                dtype=np.float32,
            )

            print(
                "Using saved crop points."
            )

        else:
            rgb_points = (
                select_four_corners(
                    source_rgb,
                    (
                        f"RGB | "
                        f"{day} | "
                        f"{tray}"
                    ),
                    grayscale=False,
                )
            )

            if rgb_points is None:
                record[
                    "status"
                ] = "CANCELLED"

                record[
                    "notes"
                ] = (
                    "RGB crop selection "
                    "was cancelled."
                )

                crop_manifest_records.append(
                    record
                )

                print(
                    "Result: CANCELLED"
                )

                continue

            ms_points = (
                select_four_corners(
                    display_stretch(
                        source_nir
                    ),
                    (
                        f"MS-NIR | "
                        f"{day} | "
                        f"{tray}"
                    ),
                    grayscale=True,
                )
            )

            if ms_points is None:
                record[
                    "status"
                ] = "CANCELLED"

                record[
                    "notes"
                ] = (
                    "NIR crop selection "
                    "was cancelled."
                )

                crop_manifest_records.append(
                    record
                )

                print(
                    "Result: CANCELLED"
                )

                continue

            crop_points[crop_key] = {
                "day_order": (
                    day_number
                ),
                "tray_no": (
                    tray_number
                ),
                "rgb_points": (
                    rgb_points.tolist()
                ),
                "ms_points": (
                    ms_points.tolist()
                ),
            }

        # ====================================================
        # SAVE RGB CROP
        # ====================================================

        rgb_crop = warp_to_standard(
            source_rgb,
            rgb_points,
            cv2.INTER_LINEAR,
        )

        Image.fromarray(
            rgb_crop
        ).save(
            outputs["D"],
            quality=100,
            subsampling=0,
        )

        # ====================================================
        # SAVE MULTISPECTRAL CROPS
        # ====================================================

        for band in [
            "MS_G",
            "MS_R",
            "MS_RE",
            "MS_NIR",
        ]:
            source_band_path = Path(
                row[
                    f"source_{band.lower()}"
                ]
            )

            source_band = read_band(
                source_band_path
            )

            band_crop = warp_to_standard(
                source_band,
                ms_points,
                cv2.INTER_NEAREST,
            )

            tifffile.imwrite(
                outputs[band],
                band_crop,
            )

        record[
            "status"
        ] = "PASS"

        record[
            "notes"
        ] = (
            "RGB and all four multispectral "
            "bands were cropped."
        )

        print("Result: PASS")

    except Exception as error:
        record[
            "status"
        ] = "FAIL"

        record[
            "notes"
        ] = str(error)

        print("Result: FAIL")
        print(error)

    crop_manifest_records.append(
        record
    )

    # Save points after every completed tray.
    with open(
        crop_points_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            crop_points,
            file,
            indent=2,
        )

# ============================================================
# SAVE CROP MANIFEST
# ============================================================

crop_manifest_df = pd.DataFrame(
    crop_manifest_records
)

crop_manifest_df = (
    crop_manifest_df
    .merge(
        tray_design_df,
        on=[
            "tray_no",
            "tray",
        ],
        how="left",
    )
)

crop_manifest_df[
    "actual_environment"
] = crop_manifest_df.apply(
    lambda row: actual_environment(
        row["day_order"],
        row["stress_type"],
        row["assigned_environment"],
    ),
    axis=1,
)

crop_manifest_df[
    "watering_regime"
] = crop_manifest_df[
    "stress_type"
].map(watering_regime)

crop_manifest_df.to_csv(
    S01
    / "_reports"
    / "crop_manifest.csv",
    index=False,
)

with open(
    crop_points_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        crop_points,
        file,
        indent=2,
    )

print("\nCrop status:")

display(
    crop_manifest_df[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame()
)

failed_crops = crop_manifest_df[
    ~crop_manifest_df[
        "status"
    ].isin(
        [
            "PASS",
            "SKIPPED_EXISTING",
        ]
    )
]

if not failed_crops.empty:
    print(
        "\nCrops requiring attention:"
    )

    display(
        failed_crops[
            [
                "day",
                "tray",
                "status",
                "notes",
            ]
        ]
    )

print("\nCELL 6 COMPLETED.")


[1/12] Day 1 | Tray 1 | DJI_20260629153749_0001
Result: PASS

[2/12] Day 1 | Tray 2 | DJI_20260629153956_0003
Result: PASS

[3/12] Day 1 | Tray 3 | DJI_20260629154053_0004
Result: PASS

[4/12] Day 1 | Tray 4 | DJI_20260629154141_0005
Result: PASS

[5/12] Day 1 | Tray 5 | DJI_20260629154235_0006
Result: PASS

[6/12] Day 1 | Tray 6 | DJI_20260629154321_0007
Result: PASS

[7/12] Day 1 | Tray 7 | DJI_20260629154404_0008
Result: PASS

[8/12] Day 1 | Tray 8 | DJI_20260629154446_0009
Result: PASS

[9/12] Day 1 | Tray 9 | DJI_20260629154539_0010
Result: PASS

[10/12] Day 1 | Tray 10 | DJI_20260629154609_0011
Result: PASS

[11/12] Day 1 | Tray 11 | DJI_20260629154652_0012
Result: PASS

[12/12] Day 1 | Tray 12 | DJI_20260629154739_0013
Result: PASS

Crop status:


,count
status,
PASS,12



CELL 6 COMPLETED.


In [25]:
#7 — Check crop quality and create previews

usable_crop_df = crop_manifest_df[
    crop_manifest_df[
        "status"
    ].isin(
        [
            "PASS",
            "SKIPPED_EXISTING",
        ]
    )
].copy()

if usable_crop_df.empty:
    raise RuntimeError(
        "No usable crops were found after Cell 6."
    )

qa_records = []

for _, row in usable_crop_df.iterrows():

    day_number = int(
        row["day_order"]
    )

    tray_number = int(
        row["tray_no"]
    )

    paths = crop_file_paths(
        day_number,
        tray_number,
    )

    try:
        rgb = read_rgb(
            paths["D"]
        )

        ms_g = read_band(
            paths["MS_G"]
        )

        ms_r = read_band(
            paths["MS_R"]
        )

        ms_re = read_band(
            paths["MS_RE"]
        )

        ms_nir = read_band(
            paths["MS_NIR"]
        )

        shapes = {
            "D": tuple(
                rgb.shape[:2]
            ),
            "MS_G": tuple(
                ms_g.shape
            ),
            "MS_R": tuple(
                ms_r.shape
            ),
            "MS_RE": tuple(
                ms_re.shape
            ),
            "MS_NIR": tuple(
                ms_nir.shape
            ),
        }

        expected_shape = (
            STANDARD_HEIGHT,
            STANDARD_WIDTH,
        )

        shape_pass = all(
            shape == expected_shape
            for shape
            in shapes.values()
        )

        finite_pass = all(
            np.isfinite(
                array.astype(
                    np.float32
                )
            ).all()
            for array in [
                ms_g,
                ms_r,
                ms_re,
                ms_nir,
            ]
        )

        nonzero_pass = all(
            np.count_nonzero(
                array
            ) > 0
            for array in [
                rgb,
                ms_g,
                ms_r,
                ms_re,
                ms_nir,
            ]
        )

        if (
            shape_pass
            and finite_pass
            and nonzero_pass
        ):
            status = "PASS"
        else:
            status = "FAIL"

        preview_path = (
            S02
            / "previews"
            / (
                f"Day_{day_number:02d}"
                f"_Tray_{tray_number:02d}"
                f"_five_band_preview.png"
            )
        )

        figure, axes = plt.subplots(
            1,
            5,
            figsize=(20, 4),
        )

        axes[0].imshow(rgb)
        axes[0].set_title("RGB")

        axes[1].imshow(
            display_stretch(
                ms_g
            ),
            cmap="gray",
        )
        axes[1].set_title("Green")

        axes[2].imshow(
            display_stretch(
                ms_r
            ),
            cmap="gray",
        )
        axes[2].set_title("Red")

        axes[3].imshow(
            display_stretch(
                ms_re
            ),
            cmap="gray",
        )
        axes[3].set_title(
            "Red Edge"
        )

        axes[4].imshow(
            display_stretch(
                ms_nir
            ),
            cmap="gray",
        )
        axes[4].set_title("NIR")

        for axis in axes:
            axis.axis("off")

        figure.suptitle(
            f"Day {day_number} | "
            f"Tray {tray_number}"
        )

        figure.tight_layout()

        figure.savefig(
            preview_path,
            dpi=180,
            bbox_inches="tight",
        )

        plt.close(figure)

        qa_records.append(
            {
                "day_order": (
                    day_number
                ),
                "day": row["day"],
                "tray_no": (
                    tray_number
                ),
                "tray": row["tray"],
                "shape_d": str(
                    shapes["D"]
                ),
                "shape_ms_g": str(
                    shapes["MS_G"]
                ),
                "shape_ms_r": str(
                    shapes["MS_R"]
                ),
                "shape_ms_re": str(
                    shapes["MS_RE"]
                ),
                "shape_ms_nir": str(
                    shapes["MS_NIR"]
                ),
                "shape_pass": (
                    shape_pass
                ),
                "finite_pass": (
                    finite_pass
                ),
                "nonzero_pass": (
                    nonzero_pass
                ),
                "qa_status": status,
                "preview_path": str(
                    preview_path
                ),
            }
        )

    except Exception as error:
        qa_records.append(
            {
                "day_order": (
                    day_number
                ),
                "day": row["day"],
                "tray_no": (
                    tray_number
                ),
                "tray": row["tray"],
                "shape_d": "",
                "shape_ms_g": "",
                "shape_ms_r": "",
                "shape_ms_re": "",
                "shape_ms_nir": "",
                "shape_pass": False,
                "finite_pass": False,
                "nonzero_pass": False,
                "qa_status": "FAIL",
                "preview_path": "",
                "error": str(error),
            }
        )

qa_df = pd.DataFrame(
    qa_records
)

qa_df.to_csv(
    S02
    / "_reports"
    / "crop_quality_check.csv",
    index=False,
)

print("Crop quality status:")

display(
    qa_df[
        "qa_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame()
)

if (
    qa_df["qa_status"]
    != "PASS"
).any():
    print(
        "\nFailed quality checks:"
    )

    display(
        qa_df[
            qa_df[
                "qa_status"
            ] != "PASS"
        ]
    )

print("\nCELL 7 COMPLETED.")

Crop quality status:


,count
qa_status,
PASS,12



CELL 7 COMPLETED.


In [26]:
import matplotlib.pyplot as plt

plt.close("all")

%matplotlib inline

print("Matplotlib changed to inline mode.")

Matplotlib changed to inline mode.


In [27]:
# 8 — Create the fixed 7 × 10 cell-grid overlays

grid_records = []

passed_qa_df = qa_df[
    qa_df[
        "qa_status"
    ] == "PASS"
].copy()

for _, row in passed_qa_df.iterrows():

    day_number = int(
        row["day_order"]
    )

    tray_number = int(
        row["tray_no"]
    )

    paths = crop_file_paths(
        day_number,
        tray_number,
    )

    rgb = read_rgb(
        paths["D"]
    )

    nir = read_band(
        paths["MS_NIR"]
    )

    rgb_overlay_path = (
        S03
        / "overlays"
        / (
            f"Day_{day_number:02d}"
            f"_Tray_{tray_number:02d}"
            f"_RGB_grid.png"
        )
    )

    nir_overlay_path = (
        S06
        / "overlays"
        / (
            f"Day_{day_number:02d}"
            f"_Tray_{tray_number:02d}"
            f"_NIR_grid.png"
        )
    )

    # ========================================================
    # RGB GRID
    # ========================================================

    figure, axis = plt.subplots(
        figsize=(12, 8.4)
    )

    axis.imshow(rgb)
    add_grid(axis)

    for row_index in range(ROWS):
        for column_index in range(COLS):

            (
                x0,
                y0,
                x1,
                y1,
            ) = cell_bounds(
                row_index,
                column_index,
                inner_margin=False,
            )

            cell_number = (
                row_index
                * COLS
                + column_index
                + 1
            )

            axis.text(
                (
                    x0 + x1
                ) / 2,
                (
                    y0 + y1
                ) / 2,
                str(cell_number),
                ha="center",
                va="center",
                fontsize=6,
            )

    axis.set_title(
        f"RGB grid | "
        f"Day {day_number} | "
        f"Tray {tray_number}"
    )

    axis.axis("off")
    figure.tight_layout()

    figure.savefig(
        rgb_overlay_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(figure)

    # ========================================================
    # NIR GRID
    # ========================================================

    figure, axis = plt.subplots(
        figsize=(12, 8.4)
    )

    axis.imshow(
        display_stretch(
            nir
        ),
        cmap="gray",
    )

    add_grid(axis)

    axis.set_title(
        f"NIR grid | "
        f"Day {day_number} | "
        f"Tray {tray_number}"
    )

    axis.axis("off")
    figure.tight_layout()

    figure.savefig(
        nir_overlay_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(figure)

    grid_records.append(
        {
            "day_order": (
                day_number
            ),
            "day": row["day"],
            "tray_no": (
                tray_number
            ),
            "tray": row["tray"],
            "rows": ROWS,
            "columns": COLS,
            "cells": (
                EXPECTED_CELLS
            ),
            "rgb_grid_overlay": str(
                rgb_overlay_path
            ),
            "nir_grid_overlay": str(
                nir_overlay_path
            ),
        }
    )

grid_manifest_df = pd.DataFrame(
    grid_records
)

grid_manifest_df.to_csv(
    S03
    / "_reports"
    / "cell_grid_manifest.csv",
    index=False,
)

print(
    "Grid overlays created:",
    len(grid_manifest_df),
)

print("\nCELL 8 COMPLETED.")

Grid overlays created: 12

CELL 8 COMPLETED.


In [29]:
#9 RGB green-cover and emergence-proxy analysis

def make_green_mask(rgb):
    """
    Create a simple green vegetation mask from the RGB image.
    """

    rgb_uint8 = rgb.astype(
        np.uint8
    )

    hsv = cv2.cvtColor(
        rgb_uint8,
        cv2.COLOR_RGB2HSV,
    )

    hsv_mask = cv2.inRange(
        hsv,
        np.array(
            [
                GREEN_H_MIN,
                GREEN_S_MIN,
                GREEN_V_MIN,
            ],
            dtype=np.uint8,
        ),
        np.array(
            [
                GREEN_H_MAX,
                255,
                255,
            ],
            dtype=np.uint8,
        ),
    ) > 0

    rgb_float = rgb_uint8.astype(
        np.float32
    )

    red = rgb_float[:, :, 0]
    green = rgb_float[:, :, 1]
    blue = rgb_float[:, :, 2]

    excess_green = (
        2.0 * green
        - red
        - blue
    )

    green_mask = (
        hsv_mask
        & (
            excess_green
            >= EXCESS_GREEN_MIN
        )
    )

    return green_mask


rgb_cell_records = []
rgb_tray_records = []

for _, row in passed_qa_df.iterrows():

    day_number = int(
        row["day_order"]
    )

    tray_number = int(
        row["tray_no"]
    )

    paths = crop_file_paths(
        day_number,
        tray_number,
    )

    rgb = read_rgb(
        paths["D"]
    )

    green_mask = make_green_mask(
        rgb
    )

    tray_cell_records = []

    for row_index in range(ROWS):
        for column_index in range(COLS):

            (
                x0,
                y0,
                x1,
                y1,
            ) = cell_bounds(
                row_index,
                column_index,
                inner_margin=True,
            )

            cell_mask = green_mask[
                y0:y1,
                x0:x1,
            ]

            green_pixels = int(
                np.count_nonzero(
                    cell_mask
                )
            )

            total_pixels = int(
                cell_mask.size
            )

            if total_pixels > 0:
                green_cover_pct = (
                    100.0
                    * green_pixels
                    / total_pixels
                )
            else:
                green_cover_pct = np.nan

            emerged_proxy = int(
                green_cover_pct
                >= EMERGED_GREEN_COVER_PCT
            )

            cell_number = (
                row_index
                * COLS
                + column_index
                + 1
            )

            cell_record = {
                "day_order": (
                    day_number
                ),
                "day": row["day"],
                "tray_no": (
                    tray_number
                ),
                "tray": row["tray"],
                "row": (
                    row_index + 1
                ),
                "column": (
                    column_index + 1
                ),
                "cell_number": (
                    cell_number
                ),
                "green_pixels": (
                    green_pixels
                ),
                "cell_pixels": (
                    total_pixels
                ),
                "green_cover_pct": (
                    green_cover_pct
                ),
                "emerged_proxy": (
                    emerged_proxy
                ),
            }

            tray_cell_records.append(
                cell_record
            )

            rgb_cell_records.append(
                cell_record
            )

    tray_cell_df = pd.DataFrame(
        tray_cell_records
    )

    rgb_tray_records.append(
        {
            "day_order": (
                day_number
            ),
            "day": row["day"],
            "tray_no": (
                tray_number
            ),
            "tray": row["tray"],
            "emerged_cells_proxy": int(
                tray_cell_df[
                    "emerged_proxy"
                ].sum()
            ),
            "emergence_pct_proxy": float(
                100.0
                * tray_cell_df[
                    "emerged_proxy"
                ].mean()
            ),
            "mean_green_cover_pct": float(
                tray_cell_df[
                    "green_cover_pct"
                ].mean()
            ),
            "median_green_cover_pct": float(
                tray_cell_df[
                    "green_cover_pct"
                ].median()
            ),
            "total_green_pixels": int(
                tray_cell_df[
                    "green_pixels"
                ].sum()
            ),
        }
    )

    # ========================================================
    # SAVE CELL OVERLAY
    # ========================================================

    overlay_path = (
        S04
        / "overlays"
        / (
            f"Day_{day_number:02d}"
            f"_Tray_{tray_number:02d}"
            f"_emergence_overlay.png"
        )
    )

    figure, axis = plt.subplots(
        figsize=(12, 8.4)
    )

    axis.imshow(rgb)

    for cell_record in tray_cell_records:

        (
            x0,
            y0,
            x1,
            y1,
        ) = cell_bounds(
            cell_record["row"] - 1,
            cell_record["column"] - 1,
            inner_margin=False,
        )

        axis.add_patch(
            plt.Rectangle(
                (
                    x0,
                    y0,
                ),
                x1 - x0,
                y1 - y0,
                fill=False,
                linewidth=0.8,
            )
        )

        axis.text(
            (
                x0 + x1
            ) / 2,
            (
                y0 + y1
            ) / 2,
            (
                f"{cell_record['green_cover_pct']:.1f}"
            ),
            ha="center",
            va="center",
            fontsize=5,
        )

    axis.set_title(
        "RGB green-cover proxy | "
        f"Day {day_number} | "
        f"Tray {tray_number}"
    )

    axis.axis("off")
    figure.tight_layout()

    figure.savefig(
        overlay_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(figure)

# ============================================================
# CREATE RGB TABLES
# ============================================================

rgb_cell_df = pd.DataFrame(
    rgb_cell_records
)

rgb_tray_df = pd.DataFrame(
    rgb_tray_records
)

rgb_cell_df = rgb_cell_df.merge(
    tray_design_df,
    on=[
        "tray_no",
        "tray",
    ],
    how="left",
)

rgb_tray_df = rgb_tray_df.merge(
    tray_design_df,
    on=[
        "tray_no",
        "tray",
    ],
    how="left",
)

for dataframe in [
    rgb_cell_df,
    rgb_tray_df,
]:
    dataframe[
        "actual_environment"
    ] = dataframe.apply(
        lambda row: actual_environment(
            row["day_order"],
            row["stress_type"],
            row["assigned_environment"],
        ),
        axis=1,
    )

    dataframe[
        "watering_regime"
    ] = dataframe[
        "stress_type"
    ].map(watering_regime)

rgb_tray_df = (
    rgb_tray_df
    .sort_values(
        [
            "tray_no",
            "day_order",
        ]
    )
    .reset_index(drop=True)
)

# Estimated number of cells newly visible each day.
rgb_tray_df[
    "newly_emerged_today_proxy"
] = (
    rgb_tray_df
    .groupby(
        "tray_no"
    )[
        "emerged_cells_proxy"
    ]
    .diff()
    .fillna(
        rgb_tray_df[
            "emerged_cells_proxy"
        ]
    )
    .clip(lower=0)
)

# ============================================================
# SAVE RGB RESULTS
# ============================================================

rgb_cell_df.to_csv(
    S04
    / "_reports"
    / "rgb_cell_metrics.csv",
    index=False,
)

rgb_tray_df.to_csv(
    S04
    / "_reports"
    / "rgb_tray_summary.csv",
    index=False,
)

save_excel_sheets(
    S04
    / "_reports"
    / "third_trial_rgb_analysis.xlsx",
    {
        "Cell Metrics": (
            rgb_cell_df
        ),
        "Tray Summary": (
            rgb_tray_df
        ),
    },
)

print(
    "RGB cell observations:",
    len(rgb_cell_df),
)

print(
    "RGB tray/day summaries:",
    len(rgb_tray_df),
)

display(
    rgb_tray_df.head(20)
)

print("\nCELL 9 COMPLETED.")

RGB cell observations: 840
RGB tray/day summaries: 12


,day_order,day,tray_no,tray,emerged_cells_proxy,emergence_pct_proxy,mean_green_cover_pct,median_green_cover_pct,total_green_pixels,microbe_status,stress_type,assigned_environment,treatment_group,design_group,actual_environment,watering_regime,newly_emerged_today_proxy
0,1,Day 1,1,Tray 1,0,0.000000,0.011646,0.000000,69,No Microbe,Ideal,Inside,No Microbe | Ideal,No Microbe | Ideal | Inside,Inside,Always watered,0.0
1,1,Day 1,2,Tray 2,0,0.000000,0.005063,0.000000,30,No Microbe,Ideal,Outside,No Microbe | Ideal,No Microbe | Ideal | Outside,Outside,Always watered,0.0
2,1,Day 1,3,Tray 3,0,0.000000,0.007595,0.000000,45,No Microbe,Moisture,Outside,No Microbe | Moisture,No Microbe | Moisture | Outside,Outside,Moisture treatment,0.0
3,1,Day 1,4,Tray 4,1,1.428571,0.021942,0.000000,130,No Microbe,Heat,Alternating,No Microbe | Heat,No Microbe | Heat | Alternating,Inside,Always watered,1.0
4,1,Day 1,5,Tray 5,1,1.428571,0.020254,0.000000,120,Microbe,Moisture,Inside,Microbe | Moisture,Microbe | Moisture | Inside,Inside,Moisture treatment,1.0
5,1,Day 1,6,Tray 6,1,1.428571,0.017553,0.000000,104,Microbe,Heat,Alternating,Microbe | Heat,Microbe | Heat | Alternating,Inside,Always watered,1.0
6,1,Day 1,7,Tray 7,0,0.000000,0.014515,0.000000,86,Microbe,Ideal,Outside,Microbe | Ideal,Microbe | Ideal | Outside,Outside,Always watered,0.0
7,1,Day 1,8,Tray 8,1,1.428571,0.035107,0.011815,208,Microbe,Heat,Alternating,Microbe | Heat,Microbe | Heat | Alternating,Inside,Always watered,1.0
8,1,Day 1,9,Tray 9,2,2.857143,0.039326,0.011815,233,Microbe,Ideal,Inside,Microbe | Ideal,Microbe | Ideal | Inside,Inside,Always watered,2.0
9,1,Day 1,10,Tray 10,0,0.000000,0.011815,0.000000,70,Microbe,Moisture,Outside,Microbe | Moisture,Microbe | Moisture | Outside,Outside,Moisture treatment,0.0



CELL 9 COMPLETED.


In [32]:
#10 — RGB treatment, stress and environment comparisons

def group_summary(
    dataframe,
    group_columns,
    metric_columns,
):
    """
    Calculate mean, standard deviation, minimum and maximum
    for each requested metric.
    """

    aggregations = {}

    for metric in metric_columns:
        aggregations[
            f"{metric}_mean"
        ] = (
            metric,
            "mean",
        )

        aggregations[
            f"{metric}_std"
        ] = (
            metric,
            "std",
        )

        aggregations[
            f"{metric}_min"
        ] = (
            metric,
            "min",
        )

        aggregations[
            f"{metric}_max"
        ] = (
            metric,
            "max",
        )

    aggregations[
        "n_trays"
    ] = (
        "tray_no",
        "nunique",
    )

    return (
        dataframe
        .groupby(
            group_columns,
            as_index=False,
        )
        .agg(
            **aggregations
        )
    )


rgb_group_summary_df = group_summary(
    rgb_tray_df,
    [
        "day_order",
        "day",
        "microbe_status",
        "stress_type",
        "actual_environment",
    ],
    [
        "emerged_cells_proxy",
        "emergence_pct_proxy",
        "mean_green_cover_pct",
        "newly_emerged_today_proxy",
    ],
)

rgb_group_summary_df.to_csv(
    S05
    / "_reports"
    / "rgb_group_summary.csv",
    index=False,
)


def plot_metric_by_group(
    dataframe,
    metric,
    group_column,
    output_path,
    title,
    y_label,
):
    """
    Create a separate line for every treatment group.
    """

    figure, axis = plt.subplots(
        figsize=(11, 6)
    )

    for (
        group_name,
        group_df,
    ) in dataframe.groupby(
        group_column
    ):
        plot_df = (
            group_df
            .groupby(
                "day_order",
                as_index=False,
            )[
                metric
            ]
            .mean()
            .sort_values(
                "day_order"
            )
        )

        axis.plot(
            plot_df[
                "day_order"
            ],
            plot_df[
                metric
            ],
            marker="o",
            label=str(
                group_name
            ),
        )

    axis.set_title(title)
    axis.set_xlabel("Day")
    axis.set_ylabel(y_label)

    axis.set_xticks(
        sorted(
            dataframe[
                "day_order"
            ]
            .unique()
            .tolist()
        )
    )

    axis.legend(
        fontsize=8
    )

    axis.grid(
        alpha=0.25
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.close(figure)


rgb_tray_df[
    "microbe_stress_group"
] = (
    rgb_tray_df[
        "microbe_status"
    ]
    + " | "
    + rgb_tray_df[
        "stress_type"
    ]
)

plot_metric_by_group(
    rgb_tray_df,
    "emerged_cells_proxy",
    "microbe_stress_group",
    (
        S05
        / "charts"
        / "emerged_cells_by_microbe_and_stress.png"
    ),
    (
        "Visible emergence proxy by "
        "microbe treatment and stress type"
    ),
    "Emerged cells proxy (out of 70)",
)

plot_metric_by_group(
    rgb_tray_df,
    "mean_green_cover_pct",
    "microbe_stress_group",
    (
        S05
        / "charts"
        / "green_cover_by_microbe_and_stress.png"
    ),
    (
        "Mean RGB green cover by "
        "microbe treatment and stress type"
    ),
    "Mean green cover (%)",
)

plot_metric_by_group(
    rgb_tray_df,
    "emerged_cells_proxy",
    "actual_environment",
    (
        S05
        / "charts"
        / "emerged_cells_by_actual_environment.png"
    ),
    (
        "Visible emergence proxy by "
        "actual environment"
    ),
    "Emerged cells proxy (out of 70)",
)

plot_metric_by_group(
    rgb_tray_df,
    "mean_green_cover_pct",
    "actual_environment",
    (
        S05
        / "charts"
        / "green_cover_by_actual_environment.png"
    ),
    (
        "RGB green cover by "
        "actual environment"
    ),
    "Mean green cover (%)",
)

print(
    "RGB comparison charts and tables created."
)

print("\nCELL 10 COMPLETED.")



RGB comparison charts and tables created.

CELL 10 COMPLETED.


In [36]:
#11 — Calculate NDVI, NDRE and GNDVI

def safe_index(
    numerator,
    denominator,
):
    """
    Safely calculate a normalised difference index.
    """

    numerator = numerator.astype(
        np.float32
    )

    denominator = denominator.astype(
        np.float32
    )

    result = np.full(
        numerator.shape,
        np.nan,
        dtype=np.float32,
    )

    valid = (
        np.isfinite(
            numerator
        )
        & np.isfinite(
            denominator
        )
    )

    valid &= (
        np.abs(
            denominator
        ) > 1e-6
    )

    result[valid] = (
        numerator[valid]
        / denominator[valid]
    )

    return np.clip(
        result,
        -1.0,
        1.0,
    )


def finite_summary(
    values,
    prefix,
):
    """
    Calculate summary values after excluding NaN pixels.
    """

    values = np.asarray(
        values,
        dtype=np.float32,
    )

    valid = values[
        np.isfinite(values)
    ]

    if valid.size == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
        }

    return {
        f"{prefix}_mean": float(
            np.mean(valid)
        ),
        f"{prefix}_median": float(
            np.median(valid)
        ),
        f"{prefix}_std": float(
            np.std(valid)
        ),
        f"{prefix}_min": float(
            np.min(valid)
        ),
        f"{prefix}_max": float(
            np.max(valid)
        ),
    }


ms_cell_records = []
ms_tray_records = []

for _, row in passed_qa_df.iterrows():

    day_number = int(
        row["day_order"]
    )

    tray_number = int(
        row["tray_no"]
    )

    paths = crop_file_paths(
        day_number,
        tray_number,
    )

    green = read_band(
        paths["MS_G"]
    ).astype(np.float32)

    red = read_band(
        paths["MS_R"]
    ).astype(np.float32)

    red_edge = read_band(
        paths["MS_RE"]
    ).astype(np.float32)

    nir = read_band(
        paths["MS_NIR"]
    ).astype(np.float32)

    # ========================================================
    # VEGETATION INDICES
    # ========================================================

    ndvi = safe_index(
        nir - red,
        nir + red,
    )

    ndre = safe_index(
        nir - red_edge,
        nir + red_edge,
    )

    gndvi = safe_index(
        nir - green,
        nir + green,
    )

    tray_cell_records = []

    for row_index in range(ROWS):
        for column_index in range(COLS):

            (
                x0,
                y0,
                x1,
                y1,
            ) = cell_bounds(
                row_index,
                column_index,
                inner_margin=True,
            )

            cell_green = green[
                y0:y1,
                x0:x1,
            ]

            cell_red = red[
                y0:y1,
                x0:x1,
            ]

            cell_red_edge = red_edge[
                y0:y1,
                x0:x1,
            ]

            cell_nir = nir[
                y0:y1,
                x0:x1,
            ]

            cell_ndvi = ndvi[
                y0:y1,
                x0:x1,
            ]

            cell_ndre = ndre[
                y0:y1,
                x0:x1,
            ]

            cell_gndvi = gndvi[
                y0:y1,
                x0:x1,
            ]

            cell_number = (
                row_index
                * COLS
                + column_index
                + 1
            )

            record = {
                "day_order": (
                    day_number
                ),
                "day": row["day"],
                "tray_no": (
                    tray_number
                ),
                "tray": row["tray"],
                "row": (
                    row_index + 1
                ),
                "column": (
                    column_index + 1
                ),
                "cell_number": (
                    cell_number
                ),
                "ms_g_mean": float(
                    np.nanmean(
                        cell_green
                    )
                ),
                "ms_r_mean": float(
                    np.nanmean(
                        cell_red
                    )
                ),
                "ms_re_mean": float(
                    np.nanmean(
                        cell_red_edge
                    )
                ),
                "ms_nir_mean": float(
                    np.nanmean(
                        cell_nir
                    )
                ),
                "ndvi_vegetation_pct_proxy": float(
                    100.0
                    * np.nanmean(
                        cell_ndvi
                        > NDVI_VEGETATION_THRESHOLD
                    )
                ),
            }

            record.update(
                finite_summary(
                    cell_ndvi,
                    "ndvi",
                )
            )

            record.update(
                finite_summary(
                    cell_ndre,
                    "ndre",
                )
            )

            record.update(
                finite_summary(
                    cell_gndvi,
                    "gndvi",
                )
            )

            tray_cell_records.append(
                record
            )

            ms_cell_records.append(
                record
            )

    tray_cell_df = pd.DataFrame(
        tray_cell_records
    )

    ms_tray_records.append(
        {
            "day_order": (
                day_number
            ),
            "day": row["day"],
            "tray_no": (
                tray_number
            ),
            "tray": row["tray"],
            "mean_ndvi": float(
                tray_cell_df[
                    "ndvi_mean"
                ].mean()
            ),
            "median_ndvi": float(
                tray_cell_df[
                    "ndvi_median"
                ].median()
            ),
            "mean_ndre": float(
                tray_cell_df[
                    "ndre_mean"
                ].mean()
            ),
            "median_ndre": float(
                tray_cell_df[
                    "ndre_median"
                ].median()
            ),
            "mean_gndvi": float(
                tray_cell_df[
                    "gndvi_mean"
                ].mean()
            ),
            "median_gndvi": float(
                tray_cell_df[
                    "gndvi_median"
                ].median()
            ),
            "mean_ndvi_vegetation_pct_proxy": float(
                tray_cell_df[
                    "ndvi_vegetation_pct_proxy"
                ].mean()
            ),
            "between_cell_ndvi_std": float(
                tray_cell_df[
                    "ndvi_mean"
                ].std()
            ),
            "between_cell_ndre_std": float(
                tray_cell_df[
                    "ndre_mean"
                ].std()
            ),
            "between_cell_gndvi_std": float(
                tray_cell_df[
                    "gndvi_mean"
                ].std()
            ),
        }
    )

    # ========================================================
    # SAVE INDEX HEATMAP
    # ========================================================

    heatmap_path = (
        S07
        / "heatmaps"
        / (
            f"Day_{day_number:02d}"
            f"_Tray_{tray_number:02d}"
            f"_indices.png"
        )
    )

    figure, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4.5),
    )

    images = [
        axes[0].imshow(
            ndvi,
            vmin=-1,
            vmax=1,
        ),
        axes[1].imshow(
            ndre,
            vmin=-1,
            vmax=1,
        ),
        axes[2].imshow(
            gndvi,
            vmin=-1,
            vmax=1,
        ),
    ]

    titles = [
        "NDVI",
        "NDRE",
        "GNDVI",
    ]

    for (
        axis,
        image_object,
        title,
    ) in zip(
        axes,
        images,
        titles,
    ):
        axis.set_title(title)
        axis.axis("off")

        figure.colorbar(
            image_object,
            ax=axis,
            fraction=0.046,
            pad=0.04,
        )

    figure.suptitle(
        f"Relative indices | "
        f"Day {day_number} | "
        f"Tray {tray_number}"
    )

    figure.tight_layout()

    figure.savefig(
        heatmap_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(figure)

# ============================================================
# CREATE MULTISPECTRAL TABLES
# ============================================================

ms_cell_df = pd.DataFrame(
    ms_cell_records
)

ms_tray_df = pd.DataFrame(
    ms_tray_records
)

ms_cell_df = ms_cell_df.merge(
    tray_design_df,
    on=[
        "tray_no",
        "tray",
    ],
    how="left",
)

ms_tray_df = ms_tray_df.merge(
    tray_design_df,
    on=[
        "tray_no",
        "tray",
    ],
    how="left",
)

for dataframe in [
    ms_cell_df,
    ms_tray_df,
]:
    dataframe[
        "actual_environment"
    ] = dataframe.apply(
        lambda row: actual_environment(
            row["day_order"],
            row["stress_type"],
            row["assigned_environment"],
        ),
        axis=1,
    )

    dataframe[
        "watering_regime"
    ] = dataframe[
        "stress_type"
    ].map(watering_regime)

# ============================================================
# SAVE MULTISPECTRAL RESULTS
# ============================================================

ms_cell_df.to_csv(
    S07
    / "_reports"
    / "ms_cell_metrics.csv",
    index=False,
)

ms_tray_df.to_csv(
    S07
    / "_reports"
    / "ms_tray_summary.csv",
    index=False,
)

save_excel_sheets(
    S07
    / "_reports"
    / "third_trial_multispectral_analysis.xlsx",
    {
        "Cell Metrics": (
            ms_cell_df
        ),
        "Tray Summary": (
            ms_tray_df
        ),
    },
)

print(
    "Multispectral cell observations:",
    len(ms_cell_df),
)

print(
    "Multispectral tray/day summaries:",
    len(ms_tray_df),
)

display(
    ms_tray_df.head(20)
)

print("\nCELL 11 COMPLETED.")

Multispectral cell observations: 840
Multispectral tray/day summaries: 12


,day_order,day,tray_no,tray,mean_ndvi,median_ndvi,mean_ndre,median_ndre,mean_gndvi,median_gndvi,...,between_cell_ndvi_std,between_cell_ndre_std,between_cell_gndvi_std,microbe_status,stress_type,assigned_environment,treatment_group,design_group,actual_environment,watering_regime
0,1,Day 1,1,Tray 1,0.313045,0.296960,-0.035164,-0.083564,0.131954,0.064666,...,0.064453,0.055642,0.118332,No Microbe,Ideal,Inside,No Microbe | Ideal,No Microbe | Ideal | Inside,Inside,Always watered
1,1,Day 1,2,Tray 2,0.304542,0.246213,-0.053214,-0.094047,0.094610,-0.045846,...,0.053041,0.045380,0.107627,No Microbe,Ideal,Outside,No Microbe | Ideal,No Microbe | Ideal | Outside,Outside,Always watered
2,1,Day 1,3,Tray 3,0.304678,0.256900,-0.060774,-0.107337,0.086647,-0.034138,...,0.055468,0.047417,0.109708,No Microbe,Moisture,Outside,No Microbe | Moisture,No Microbe | Moisture | Outside,Outside,Moisture treatment
3,1,Day 1,4,Tray 4,0.316519,0.254868,-0.047943,-0.099510,0.098202,-0.042843,...,0.056503,0.054898,0.109772,No Microbe,Heat,Alternating,No Microbe | Heat,No Microbe | Heat | Alternating,Inside,Always watered
4,1,Day 1,5,Tray 5,0.311405,0.289134,-0.037489,-0.078786,0.105310,-0.011326,...,0.053986,0.053737,0.103297,Microbe,Moisture,Inside,Microbe | Moisture,Microbe | Moisture | Inside,Inside,Moisture treatment
5,1,Day 1,6,Tray 6,0.321103,0.343155,-0.020061,-0.067138,0.150520,0.208293,...,0.066980,0.058541,0.116280,Microbe,Heat,Alternating,Microbe | Heat,Microbe | Heat | Alternating,Inside,Always watered
6,1,Day 1,7,Tray 7,0.325655,0.361464,-0.029058,-0.054308,0.160532,0.212331,...,0.063523,0.051112,0.111406,Microbe,Ideal,Outside,Microbe | Ideal,Microbe | Ideal | Outside,Outside,Always watered
7,1,Day 1,8,Tray 8,0.304785,0.307430,-0.051690,-0.096416,0.104869,0.033899,...,0.057409,0.053271,0.106597,Microbe,Heat,Alternating,Microbe | Heat,Microbe | Heat | Alternating,Inside,Always watered
8,1,Day 1,9,Tray 9,0.328593,0.353649,-0.016501,-0.059748,0.155229,0.219958,...,0.070215,0.063086,0.110023,Microbe,Ideal,Inside,Microbe | Ideal,Microbe | Ideal | Inside,Inside,Always watered
9,1,Day 1,10,Tray 10,0.308799,0.313983,-0.035948,-0.086280,0.120448,0.101758,...,0.060115,0.051570,0.118355,Microbe,Moisture,Outside,Microbe | Moisture,Microbe | Moisture | Outside,Outside,Moisture treatment



CELL 11 COMPLETED.


In [38]:
#12 — Multispectral comparisons

ms_group_summary_df = group_summary(
    ms_tray_df,
    [
        "day_order",
        "day",
        "microbe_status",
        "stress_type",
        "actual_environment",
    ],
    [
        "mean_ndvi",
        "mean_ndre",
        "mean_gndvi",
        "mean_ndvi_vegetation_pct_proxy",
    ],
)

ms_group_summary_df.to_csv(
    S08
    / "_reports"
    / "ms_group_summary.csv",
    index=False,
)

ms_tray_df[
    "microbe_stress_group"
] = (
    ms_tray_df[
        "microbe_status"
    ]
    + " | "
    + ms_tray_df[
        "stress_type"
    ]
)

for metric, label in [
    (
        "mean_ndvi",
        "Mean relative NDVI",
    ),
    (
        "mean_ndre",
        "Mean relative NDRE",
    ),
    (
        "mean_gndvi",
        "Mean relative GNDVI",
    ),
]:
    plot_metric_by_group(
        ms_tray_df,
        metric,
        "microbe_stress_group",
        (
            S08
            / "charts"
            / (
                f"{metric}"
                f"_by_microbe_and_stress.png"
            )
        ),
        (
            f"{label} by microbe "
            "treatment and stress type"
        ),
        label,
    )

    plot_metric_by_group(
        ms_tray_df,
        metric,
        "actual_environment",
        (
            S08
            / "charts"
            / (
                f"{metric}"
                f"_by_actual_environment.png"
            )
        ),
        (
            f"{label} by actual environment"
        ),
        label,
    )

print(
    "Multispectral comparison charts "
    "and tables created."
)

print("\nCELL 12 COMPLETED.")

Multispectral comparison charts and tables created.

CELL 12 COMPLETED.


In [40]:
#13 — Merge RGB and multispectral results

master_df = rgb_tray_df.merge(
    ms_tray_df[
        [
            "day_order",
            "day",
            "tray_no",
            "tray",
            "mean_ndvi",
            "median_ndvi",
            "mean_ndre",
            "median_ndre",
            "mean_gndvi",
            "median_gndvi",
            "mean_ndvi_vegetation_pct_proxy",
            "between_cell_ndvi_std",
            "between_cell_ndre_std",
            "between_cell_gndvi_std",
        ]
    ],
    on=[
        "day_order",
        "day",
        "tray_no",
        "tray",
    ],
    how="inner",
)

master_df = (
    master_df
    .sort_values(
        [
            "day_order",
            "tray_no",
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# RGB–MULTISPECTRAL CORRELATIONS
# ============================================================

correlation_columns = [
    "emerged_cells_proxy",
    "emergence_pct_proxy",
    "mean_green_cover_pct",
    "mean_ndvi",
    "mean_ndre",
    "mean_gndvi",
    "mean_ndvi_vegetation_pct_proxy",
]

correlation_df = master_df[
    correlation_columns
].corr(
    method="pearson"
)

correlation_df.to_csv(
    S09
    / "_reports"
    / "rgb_ms_correlations.csv"
)

# ============================================================
# DAILY AND OVERALL SUMMARIES
# ============================================================

daily_synthesis_df = group_summary(
    master_df,
    [
        "day_order",
        "day",
        "microbe_status",
        "stress_type",
        "actual_environment",
    ],
    [
        "emerged_cells_proxy",
        "mean_green_cover_pct",
        "mean_ndvi",
        "mean_ndre",
        "mean_gndvi",
    ],
)

overall_group_summary_df = group_summary(
    master_df,
    [
        "microbe_status",
        "stress_type",
        "assigned_environment",
    ],
    [
        "emerged_cells_proxy",
        "mean_green_cover_pct",
        "mean_ndvi",
        "mean_ndre",
        "mean_gndvi",
    ],
)

# ============================================================
# FINAL-DAY RANKING
# ============================================================

last_day_number = int(
    master_df[
        "day_order"
    ].max()
)

final_day_df = master_df[
    master_df[
        "day_order"
    ] == last_day_number
].copy()

final_day_ranking_df = (
    final_day_df
    .sort_values(
        [
            "emerged_cells_proxy",
            "mean_green_cover_pct",
            "mean_ndvi",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

# ============================================================
# SAVE SYNTHESIS RESULTS
# ============================================================

master_df.to_csv(
    S09
    / "_reports"
    / "third_trial_master_summary.csv",
    index=False,
)

daily_synthesis_df.to_csv(
    S09
    / "_reports"
    / "third_trial_daily_synthesis.csv",
    index=False,
)

overall_group_summary_df.to_csv(
    S09
    / "_reports"
    / "third_trial_overall_group_summary.csv",
    index=False,
)

final_day_ranking_df.to_csv(
    S09
    / "_reports"
    / "third_trial_final_day_ranking.csv",
    index=False,
)

save_excel_sheets(
    S09
    / "_reports"
    / "third_trial_complete_results.xlsx",
    {
        "Master Summary": (
            master_df
        ),
        "Daily Synthesis": (
            daily_synthesis_df
        ),
        "Overall Groups": (
            overall_group_summary_df
        ),
        "Final Day Ranking": (
            final_day_ranking_df
        ),
        "RGB Group Summary": (
            rgb_group_summary_df
        ),
        "MS Group Summary": (
            ms_group_summary_df
        ),
        "Tray Design": (
            tray_design_df
        ),
        "Day Completeness": (
            day_completeness_df
        ),
    },
)

print(
    "Master rows:",
    len(master_df),
)

print(
    "Last analysed day:",
    last_day_number,
)

display(
    master_df.head(20)
)

print("\nCELL 13 COMPLETED.")

Master rows: 12
Last analysed day: 1


,day_order,day,tray_no,tray,emerged_cells_proxy,emergence_pct_proxy,mean_green_cover_pct,median_green_cover_pct,total_green_pixels,microbe_status,...,mean_ndvi,median_ndvi,mean_ndre,median_ndre,mean_gndvi,median_gndvi,mean_ndvi_vegetation_pct_proxy,between_cell_ndvi_std,between_cell_ndre_std,between_cell_gndvi_std
0,1,Day 1,1,Tray 1,0,0.000000,0.011646,0.000000,69,No Microbe,...,0.313045,0.296960,-0.035164,-0.083564,0.131954,0.064666,70.639346,0.064453,0.055642,0.118332
1,1,Day 1,2,Tray 2,0,0.000000,0.005063,0.000000,30,No Microbe,...,0.304542,0.246213,-0.053214,-0.094047,0.094610,-0.045846,76.662335,0.053041,0.045380,0.107627
2,1,Day 1,3,Tray 3,0,0.000000,0.007595,0.000000,45,No Microbe,...,0.304678,0.256900,-0.060774,-0.107337,0.086647,-0.034138,73.511005,0.055468,0.047417,0.109708
3,1,Day 1,4,Tray 4,1,1.428571,0.021942,0.000000,130,No Microbe,...,0.316519,0.254868,-0.047943,-0.099510,0.098202,-0.042843,80.274440,0.056503,0.054898,0.109772
4,1,Day 1,5,Tray 5,1,1.428571,0.020254,0.000000,120,Microbe,...,0.311405,0.289134,-0.037489,-0.078786,0.105310,-0.011326,70.318661,0.053986,0.053737,0.103297
5,1,Day 1,6,Tray 6,1,1.428571,0.017553,0.000000,104,Microbe,...,0.321103,0.343155,-0.020061,-0.067138,0.150520,0.208293,69.699905,0.066980,0.058541,0.116280
6,1,Day 1,7,Tray 7,0,0.000000,0.014515,0.000000,86,Microbe,...,0.325655,0.361464,-0.029058,-0.054308,0.160532,0.212331,72.603463,0.063523,0.051112,0.111406
7,1,Day 1,8,Tray 8,1,1.428571,0.035107,0.011815,208,Microbe,...,0.304785,0.307430,-0.051690,-0.096416,0.104869,0.033899,68.858527,0.057409,0.053271,0.106597
8,1,Day 1,9,Tray 9,2,2.857143,0.039326,0.011815,233,Microbe,...,0.328593,0.353649,-0.016501,-0.059748,0.155229,0.219958,72.160917,0.070215,0.063086,0.110023
9,1,Day 1,10,Tray 10,0,0.000000,0.011815,0.000000,70,Microbe,...,0.308799,0.313983,-0.035948,-0.086280,0.120448,0.101758,67.525149,0.060115,0.051570,0.118355



CELL 13 COMPLETED.


In [42]:
#14 — Calculate Microbe minus No-Microbe effects

effect_metrics = [
    "emerged_cells_proxy",
    "mean_green_cover_pct",
    "mean_ndvi",
    "mean_ndre",
    "mean_gndvi",
]

effect_records = []

comparison_groups = [
    "day_order",
    "day",
    "stress_type",
    "actual_environment",
]

for (
    group_values,
    group_df,
) in master_df.groupby(
    comparison_groups
):
    if not isinstance(
        group_values,
        tuple,
    ):
        group_values = (
            group_values,
        )

    group_info = dict(
        zip(
            comparison_groups,
            group_values,
        )
    )

    microbe_df = group_df[
        group_df[
            "microbe_status"
        ] == "Microbe"
    ]

    no_microbe_df = group_df[
        group_df[
            "microbe_status"
        ] == "No Microbe"
    ]

    if (
        microbe_df.empty
        or no_microbe_df.empty
    ):
        continue

    record = {
        **group_info,
        "microbe_trays": int(
            microbe_df[
                "tray_no"
            ].nunique()
        ),
        "no_microbe_trays": int(
            no_microbe_df[
                "tray_no"
            ].nunique()
        ),
    }

    for metric in effect_metrics:

        microbe_mean = float(
            microbe_df[
                metric
            ].mean()
        )

        no_microbe_mean = float(
            no_microbe_df[
                metric
            ].mean()
        )

        record[
            f"microbe_{metric}_mean"
        ] = microbe_mean

        record[
            f"no_microbe_{metric}_mean"
        ] = no_microbe_mean

        record[
            f"{metric}_difference"
        ] = (
            microbe_mean
            - no_microbe_mean
        )

    effect_records.append(
        record
    )

microbe_effect_df = pd.DataFrame(
    effect_records
)

microbe_effect_df.to_csv(
    S09
    / "_reports"
    / "microbe_minus_no_microbe_effects.csv",
    index=False,
)

if not microbe_effect_df.empty:
    display(
        microbe_effect_df.head(30)
    )

else:
    print(
        "No directly matched Microbe versus "
        "No-Microbe groups were available after "
        "grouping by stress type and actual environment."
    )

print("\nCELL 14 COMPLETED.")

,day_order,day,stress_type,actual_environment,microbe_trays,no_microbe_trays,microbe_emerged_cells_proxy_mean,no_microbe_emerged_cells_proxy_mean,emerged_cells_proxy_difference,microbe_mean_green_cover_pct_mean,...,mean_green_cover_pct_difference,microbe_mean_ndvi_mean,no_microbe_mean_ndvi_mean,mean_ndvi_difference,microbe_mean_ndre_mean,no_microbe_mean_ndre_mean,mean_ndre_difference,microbe_mean_gndvi_mean,no_microbe_mean_gndvi_mean,mean_gndvi_difference
0,1,Day 1,Heat,Inside,2,2,1.0,0.5,0.5,0.026330,...,0.007848,0.312944,0.315198,-0.002254,-0.035876,-0.047242,0.011367,0.127695,0.104925,0.022769
1,1,Day 1,Ideal,Inside,1,1,2.0,0.0,2.0,0.039326,...,0.027680,0.328593,0.313045,0.015549,-0.016501,-0.035164,0.018663,0.155229,0.131954,0.023275
2,1,Day 1,Ideal,Outside,1,1,0.0,0.0,0.0,0.014515,...,0.009452,0.325655,0.304542,0.021113,-0.029058,-0.053214,0.024156,0.160532,0.094610,0.065922
3,1,Day 1,Moisture,Inside,1,1,1.0,0.0,1.0,0.020254,...,0.012490,0.311405,0.315060,-0.003655,-0.037489,-0.054339,0.016851,0.105310,0.076803,0.028506
4,1,Day 1,Moisture,Outside,1,1,0.0,0.0,0.0,0.011815,...,0.004220,0.308799,0.304678,0.004121,-0.035948,-0.060774,0.024826,0.120448,0.086647,0.033801



CELL 14 COMPLETED.


In [44]:
#15 — Create manual emergence-validation sheet

manual_validation_df = rgb_cell_df[
    [
        "day_order",
        "day",
        "tray_no",
        "tray",
        "row",
        "column",
        "cell_number",
        "microbe_status",
        "stress_type",
        "assigned_environment",
        "actual_environment",
        "green_cover_pct",
        "emerged_proxy",
    ]
].copy()

# Enter 1 for manually confirmed germination.
# Enter 0 for manually confirmed non-germination.
manual_validation_df[
    "manual_emerged"
] = ""

manual_validation_df[
    "reviewer_notes"
] = ""

manual_validation_path = (
    S09
    / "manual_validation"
    / "manual_emergence_validation_template.csv"
)

manual_validation_df.to_csv(
    manual_validation_path,
    index=False,
)

print(
    "Manual validation template created:"
)

print(
    manual_validation_path
)

print("\nCELL 15 COMPLETED.")

Manual validation template created:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\Third Trial\09_Third_Trial_Synthesis\manual_validation\manual_emergence_validation_template.csv

CELL 15 COMPLETED.


In [46]:
#16 — Create final report package and ZIP file

PACKAGE_FIGURES = (
    S10
    / "figures"
)

PACKAGE_REPORTS = (
    S10
    / "reports"
)

# Remove only old copies from the package folders.
for folder in [
    PACKAGE_FIGURES,
    PACKAGE_REPORTS,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    for old_file in folder.iterdir():
        if old_file.is_file():
            old_file.unlink()

# ============================================================
# COPY CHARTS AND HEATMAPS
# ============================================================

figure_sources = sorted(
    list(
        (
            S05
            / "charts"
        ).glob("*.png")
    )
    + list(
        (
            S08
            / "charts"
        ).glob("*.png")
    )
    + list(
        (
            S07
            / "heatmaps"
        ).glob("*.png")
    )[:12]
)

figure_manifest_records = []

for source_path in figure_sources:

    section_name = (
        source_path
        .parent
        .parent
        .name
    )

    destination_name = (
        safe_name(
            section_name
        )
        + "_"
        + source_path.name
    )

    destination_path = (
        PACKAGE_FIGURES
        / destination_name
    )

    shutil.copy2(
        source_path,
        destination_path,
    )

    figure_manifest_records.append(
        {
            "source": str(
                source_path
            ),
            "package_file": str(
                destination_path
            ),
        }
    )

# ============================================================
# COPY TABLES AND EXCEL RESULTS
# ============================================================

report_sources = [
    (
        S01
        / "_reports"
        / "third_trial_tray_design.csv"
    ),
    (
        S01
        / "_reports"
        / "day_completeness.csv"
    ),
    (
        S02
        / "_reports"
        / "crop_quality_check.csv"
    ),
    (
        S04
        / "_reports"
        / "third_trial_rgb_analysis.xlsx"
    ),
    (
        S07
        / "_reports"
        / "third_trial_multispectral_analysis.xlsx"
    ),
    (
        S09
        / "_reports"
        / "third_trial_complete_results.xlsx"
    ),
    (
        S09
        / "_reports"
        / "third_trial_master_summary.csv"
    ),
    (
        S09
        / "_reports"
        / "third_trial_daily_synthesis.csv"
    ),
    (
        S09
        / "_reports"
        / "microbe_minus_no_microbe_effects.csv"
    ),
    (
        S09
        / "_reports"
        / "rgb_ms_correlations.csv"
    ),
    manual_validation_path,
]

for source_path in report_sources:

    if source_path.exists():
        shutil.copy2(
            source_path,
            (
                PACKAGE_REPORTS
                / source_path.name
            ),
        )

figure_manifest_df = pd.DataFrame(
    figure_manifest_records
)

figure_manifest_df.to_csv(
    S10
    / "report_figure_manifest.csv",
    index=False,
)

# ============================================================
# CREATE ZIP ARCHIVE
# ============================================================

archive_base = (
    S10
    / "Third_Trial_Report_Package"
)

archive_path = archive_base.with_suffix(
    ".zip"
)

if archive_path.exists():
    archive_path.unlink()

created_archive = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=S10,
)

print(
    "Final report package folder:"
)

print(S10)

print("\nZIP archive:")

print(created_archive)

print("\nCELL 16 COMPLETED.")

Final report package folder:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\Third Trial\10_Report_Figure_Package

ZIP archive:
C:\Users\rahma\Downloads\RINA_Internship_Analysis\outputs\Third Trial\10_Report_Figure_Package\Third_Trial_Report_Package.zip

CELL 16 COMPLETED.


In [ ]:
#17 — Final completion check and output index

all_output_files = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    ],
    key=lambda path: natural_key(
        str(path)
    ),
)

output_index_df = pd.DataFrame(
    [
        {
            "section": (
                path.relative_to(
                    OUTPUT_ROOT
                ).parts[0]
            ),
            "relative_path": str(
                path.relative_to(
                    OUTPUT_ROOT
                )
            ),
            "file_name": path.name,
            "suffix": (
                path.suffix.lower()
            ),
            "size_kb": round(
                path.stat().st_size
                / 1024,
                2,
            ),
        }
        for path in all_output_files
    ]
)

output_index_path = (
    OUTPUT_ROOT
    / "third_trial_output_index.csv"
)

output_index_df.to_csv(
    output_index_path,
    index=False,
)

completion_summary = pd.DataFrame(
    [
        {
            "accepted_days": ", ".join(
                f"Day {day}"
                for day in accepted_days
            ),
            "accepted_day_count": (
                len(accepted_days)
            ),
            "expected_trays_per_day": (
                len(EXPECTED_TRAYS)
            ),
            "analysed_day_tray_groups": (
                len(
                    analysis_manifest_df
                )
            ),
            "usable_crops": (
                len(usable_crop_df)
            ),
            "qa_passed_crops": (
                len(passed_qa_df)
            ),
            "rgb_cell_rows": (
                len(rgb_cell_df)
            ),
            "ms_cell_rows": (
                len(ms_cell_df)
            ),
            "master_summary_rows": (
                len(master_df)
            ),
            "output_file_count": (
                len(output_index_df)
            ),
            "output_root": str(
                OUTPUT_ROOT
            ),
        }
    ]
)

completion_summary.to_csv(
    OUTPUT_ROOT
    / "third_trial_completion_summary.csv",
    index=False,
)

print(
    "THIRD TRIAL ANALYSIS COMPLETED."
)

display(
    completion_summary
)

print("\nOutput index:")

print(
    output_index_path
)

print("\nMain Excel result:")

print(
    S09
    / "_reports"
    / "third_trial_complete_results.xlsx"
)

print("\nFinal ZIP package:")

print(
    S10
    / "Third_Trial_Report_Package.zip"
)